# Lab 4 — Graphs: Traversals, Shortest Path and Minimum Spanning Tree
### Advanced Algorithms and Complexity — Master in AI and Master in RSD · Chapter 3

| | |
|---|---|
| **Lectures & Tutorials** | Dr. Benyamina Ahmed |
| **Labs** | Mahkouka Mohamed Fouad |
| **Duration** | 1 session |

---
## 🎯 Objective

Implement an **undirected weighted graph** and test:
- **breadth-first (BFS)** and **depth-first (DFS)** traversals;
- **Dijkstra's** algorithm (shortest path);
- **Kruskal's** algorithm (minimum spanning tree, MST);
- the **execution time** of each operation.

> **Reminder:** an undirected graph is a set of vertices connected by edges (weighted or not). We store it as an **adjacency list**: for each vertex, the list of its neighbours with the edge weight.

**Constraints:** Python 3 only (no external library; `collections` and `heapq` are part of Python), commented code, clearly displayed results.

---
# Tools

**Run this cell first.** `check` prints ✅ or ❌; `timer_ms(f, *args)` gives the average time in **milliseconds** over 5 runs.

In [ ]:
def check(name, got, expected):
    print(("✅" if got == expected else "❌"), name, "| got :", got, "| expected :", expected)


import heapq
import random
import sys
import time
from collections import deque
from statistics import linear_regression
import math


def timer_ms(f, *args, repetitions=5):
    # Average time of f(*args) in milliseconds, over 5 runs.
    total = 0.0
    for _ in range(repetitions):
        start = time.perf_counter()
        f(*args)
        total += time.perf_counter() - start
    return total / repetitions * 1000


print("Tools ready ✅")

---
# Part 1 — The graph *(provided)*

Undirected weighted graph with 7 vertices and 11 edges; source for the traversals and Dijkstra: **A**.

| Vertex | Neighbours (weight) |
|---|---|
| A | B (4), D (8), E (1) |
| B | A (4), C (2), E (3) |
| C | B (2), E (8), F (2) |
| D | A (8), E (10) |
| E | A (1), B (3), C (8), D (10), F (7), G (6) |
| F | C (2), E (7), G (4) |
| G | E (6), F (4) |

The `Graph` class stores the adjacency list in a dictionary. Read it and run the cell.

In [ ]:
class Graph:
    def __init__(self):
        self.adj = {}           # vertex → list of (neighbour, weight)

    def add_vertex(self, s):
        if s not in self.adj:
            self.adj[s] = []

    def add_edge(self, u, v, w):
        self.add_vertex(u)
        self.add_vertex(v)
        self.adj[u].append((v, w))
        self.adj[v].append((u, w))      # undirected: both directions

    def vertices(self):
        return list(self.adj.keys())

    def edges(self):
        # each edge once, as (u, v, w)
        seen, res = set(), []
        for u in self.adj:
            for v, w in self.adj[u]:
                if (v, u) not in seen:
                    seen.add((u, v))
                    res.append((u, v, w))
        return res


g = Graph()
for u, v, w in [("A", "B", 4), ("A", "D", 8), ("A", "E", 1), ("B", "C", 2), ("B", "E", 3), ("C", "E", 8),
                ("C", "F", 2), ("D", "E", 10), ("E", "F", 7), ("E", "G", 6), ("F", "G", 4)]:
    g.add_edge(u, v, w)

for s in g.vertices():
    print(s, ":", g.adj[s])
check("number of edges", len(g.edges()), 11)

---
# Part 2 — BFS and DFS traversals

**BFS (breadth)**: visit the source, then all its neighbours, then the neighbours' neighbours… using a **queue** (`deque`): `popleft()` to dequeue, `append()` to enqueue. The **level** of a vertex is its distance in number of edges from A.

**DFS (depth)**: start from a vertex and go as far as possible before backtracking. **Recursive** version: mark `u` as seen, add it to the visit order, then visit each neighbour not yet seen.

Neighbours are always examined **in adjacency-list order**.

In [ ]:
def bfs(graph, source):
    # Returns (visit order, levels); level = distance in number of edges from source.
    visited = [source]
    levels = {source: 0}
    queue = deque([source])
    # TO COMPLETE : while the queue is not empty, dequeue u and enqueue its neighbours not yet seen
    return visited, levels


def dfs(graph, source):
    # Recursive depth-first traversal; returns the visit order.
    visited = []
    seen = set()

    def visit(u):
        # TO COMPLETE
        pass

    visit(source)
    return visited

In [ ]:
order, levels = bfs(g, "A")
check("BFS order", order, ["A", "B", "D", "E", "C", "F", "G"])
check("BFS levels", levels, {"A": 0, "B": 1, "D": 1, "E": 1, "C": 2, "F": 2, "G": 2})
check("DFS order", dfs(g, "A"), ["A", "B", "C", "E", "D", "F", "G"])

> ✍️ **Why is E at level 1 in BFS, but visited 4th in DFS?** …

---
# Part 3 — Shortest path: Dijkstra

For each vertex we keep the best known distance `dist` and its predecessor `pred`. We always extract the vertex with the **smallest distance** using a priority queue (`heapq`), then try to improve its neighbours:

```
if dist[u] + w < dist[v]:
    dist[v] ← dist[u] + w ; pred[v] ← u ; push (dist[v], v) into the queue
```

✍️ Complete the loop of `dijkstra`, then `path(pred, target)` which rebuilds the path by walking back the predecessors (then reversing the list).

In [ ]:
def dijkstra(graph, source):
    # Returns (dist, pred): minimal distance and predecessor of each vertex.
    dist = {s: float("inf") for s in graph.adj}
    pred = {s: None for s in graph.adj}
    dist[source] = 0
    pq = [(0, source)]                  # priority queue (distance, vertex)
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue                    # stale entry
        # TO COMPLETE : for each neighbour v, if d + w < dist[v]: update dist, pred, and push into pq
    return dist, pred


def path(pred, target):
    # Walks back the predecessors from target to the source.
    res = []
    # TO COMPLETE
    return res

In [ ]:
dist, pred = dijkstra(g, "A")
check("distances from A", dist, {"A": 0, "B": 4, "D": 8, "E": 1, "C": 6, "F": 8, "G": 7})
check("path A → F", path(pred, "F"), ["A", "E", "F"])
check("path A → C", path(pred, "C"), ["A", "B", "C"])

for s in g.vertices():
    print(f"A → {s} : {' → '.join(path(pred, s))}  (cost {dist[s]})")

> ✍️ **Two paths to F cost 8 (A → E → F and A → B → C → F). Why does Dijkstra keep A → E → F?** …

---
# Part 4 — Minimum spanning tree: Kruskal

A **minimum spanning tree (MST)** connects **all** vertices with $V - 1$ edges, without a cycle, for a minimal total weight.

**Kruskal**: sort the edges by increasing weight; take each edge **if it does not create a cycle**. To detect a cycle, the `UnionFind` structure (provided) groups the vertices already connected: `uf.union(u, v)` returns `False` if `u` and `v` are already in the same group.

In [ ]:
class UnionFind:
    def __init__(self, vertices):
        self.parent = {s: s for s in vertices}
        self.rank = {s: 0 for s in vertices}

    def find(self, x):                  # representative of x's set
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):              # False if x and y are already connected (cycle)
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True

In [ ]:
def kruskal(graph):
    # Returns (MST edges, total weight).
    edges = sorted(graph.edges(), key=lambda e: e[2])      # increasing weight
    uf = UnionFind(graph.vertices())
    mst, total = [], 0
    # TO COMPLETE : keep each edge (u, v, w) that does not create a cycle (uf.union returns True)
    return mst, total

In [ ]:
mst, total = kruskal(g)
for u, v, w in mst:
    print(f"{u} – {v} ({w})")
check("total MST weight", total, 20)
check("number of edges = V − 1", len(mst), len(g.vertices()) - 1)
check("D is connected by A – D (8)", ("A", "D", 8) in mst, True)

> ✍️ **Why is edge E – G (6) rejected?** …

---
# Part 5 — Execution time

## 5.1 On the studied graph

In [ ]:
print(f"{'Algorithm':<10} {'Theoretical complexity':<24} {'Average time (ms)':>18}")
for name, cplx, f in [("BFS", "O(V + E)", lambda: bfs(g, "A")), ("DFS", "O(V + E)", lambda: dfs(g, "A")),
                      ("Dijkstra", "O((V + E) log V)", lambda: dijkstra(g, "A")), ("Kruskal", "O(E log E)", lambda: kruskal(g))]:
    print(f"{name:<10} {cplx:<24} {timer_ms(f):>18.4f}")

## 5.2 On larger and larger graphs
We generate random **connected** graphs with $V$ vertices and about $E = 3V$ edges, and time each algorithm. *(Just run it.)*

In [ ]:
sys.setrecursionlimit(50_000)       # recursive DFS can go very deep

def random_graph(V, seed=0):
    rnd = random.Random(seed)
    G = Graph()
    for i in range(1, V):                                   # chain: guarantees connectivity
        G.add_edge(i - 1, i, rnd.randint(1, 100))
    for _ in range(2 * V):                                  # extra random edges
        u, v = rnd.randrange(V), rnd.randrange(V)
        if u != v:
            G.add_edge(u, v, rnd.randint(1, 100))
    return G

sizes = [100, 1_000, 5_000, 20_000]
times = {"BFS": [], "DFS": [], "Dijkstra": [], "Kruskal": []}
print(f"{'V':>7} {'E':>7} | " + " ".join(f"{k:>10}" for k in times) + "   (ms)")
for V in sizes:
    G = random_graph(V)
    for name, f in [("BFS", lambda: bfs(G, 0)), ("DFS", lambda: dfs(G, 0)),
                    ("Dijkstra", lambda: dijkstra(G, 0)), ("Kruskal", lambda: kruskal(G))]:
        times[name].append(timer_ms(f, repetitions=3))
    print(f"{V:>7,} {len(G.edges()):>7,} | " + " ".join(f"{times[k][-1]:>10.2f}" for k in times))

for name, values in times.items():
    pts = [(n, v) for n, v in zip(sizes, values) if v > 0]
    if len(pts) > 1:
        slope = linear_regression([math.log10(n) for n, _ in pts], [math.log10(v) for _, v in pts]).slope
        print(f"{name:<9} log-log slope ≈ {slope:.2f}")

> ✍️ **Do the slopes match the theoretical complexities?** …

---
# 📤 What to submit

1. This completed notebook: **Restart & Run All** with no errors, every check ✅.
2. The ✍️ lines completed.
3. Push it to GitHub:

```bash
git add tp4_graphs/
git commit -m "Lab 4: graphs"
git push
```

## 🎯 Key takeaways

| Algorithm | Purpose | Key structure | Complexity |
|---|---|---|---|
| BFS | level-by-level traversal | queue (`deque`) | $O(V + E)$ |
| DFS | depth-first traversal | recursion (stack) | $O(V + E)$ |
| Dijkstra | shortest paths (weights ≥ 0) | priority queue (`heapq`) | $O((V + E) \log V)$ |
| Kruskal | minimum spanning tree | sort + Union-Find | $O(E \log E)$ |

- Dijkstra only works with **non-negative weights**.
- An MST of $V$ vertices always has $V - 1$ edges.